## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

## Read the ingested records in table raw_data 

In [ ]:
df_raw_data = spark.sql("""
    SELECT 
        state_code,
        raw_payload,
        ingested_at
    FROM weather.raw_data
    WHERE to_date(ingested_at) = TO_DATE(TO_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), CURRENT_TIMEZONE()))
""")

## Extract the 'time' list from the JSON string

In [ ]:
df_time = (
    df_raw_data
    .withColumn(
        "time_list", 
        F.from_json(F.get_json_object("raw_payload", "$.hourly.time"), ArrayType(StringType()))
    )
)

## Transform the array into individual rows

In [ ]:
df_rows = (
    df_time
    .withColumn(
        "measurement_timestamp", 
        (F.explode(F.col("time_list")))
    )
)

## Cast and select final columns

In [ ]:
df_control = (
    df_rows
    .withColumn(
        "measurement_timestamp", 
        F.col("measurement_timestamp").cast("timestamp")
    )
    .select(
        "state_code", 
        "measurement_timestamp",
        "ingested_at"
    )
)

## Create a dataframe of expected hours per day 

In [0]:
df_expected = (
    df_control
    .select("state_code")
    .distinct()
    .withColumn(
        "hour_num", 
        F.explode(F.sequence(F.lit(0), F.lit(23)))
    )
    .withColumn(
        "expected_timestamp",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.current_date(),
                F.concat(F.lpad(F.col("hour_num"), 2, "0"), F.lit(":00:00")),
            )
        ),
    )
    .drop("hour_num")
)

## Join between expected and ingested hours to obtain the status

In [ ]:
exp = df_expected.alias("exp")
ctr = df_control.alias("ctr")

df_final = (
    exp.join(
        ctr,
        on=[
            F.col("exp.state_code") == F.col("ctr.state_code"),
            F.col("exp.expected_timestamp") == F.col("ctr.measurement_timestamp")
        ],
        how="left"
    )
    .withColumn("success_flag", F.col("ctr.state_code").isNotNull())
    .select("exp.*", "success_flag")
    .distinct()
)

## Write data in control table
Using MERGE with (state_code + expected_timestamp) as unique key

In [ ]:
table_name = "weather.ingestion_control"

if not spark.catalog.tableExists(table_name):
    (
        df_final
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_final.alias("source"), 
            condition="""
            target.state_code = source.state_code AND 
            target.expected_timestamp = source.expected_timestamp
            """
        )
        .whenMatchedUpdate(set = {"target.success_flag": "source.success_flag"})
        .whenNotMatchedInsertAll()
        .execute()
    )

## Sanity check

In [0]:
%sql
select * 
from weather.ingestion_control
order by expected_timestamp desc, state_code 
limit 10